In [ ]:
import pandas as pd
from glob import glob
import os

In [ ]:
# 1. Configuration and Paths

data_path = "../../data/raw/Dataset(raw)"
subjects_info_path = "../../data/raw/data_subjects_info.csv"

LABEL_MAP = {
    'dws': 0, 'ups': 1, 'wlk': 2, 'jog': 3, 'sit': 4, 'std': 5
}

files = sorted(glob(data_path + "/*/*.csv"))

## Quy trình xử lý dữ liệu (Preprocessing Steps)

Dưới đây là các bước thiết lập ban đầu cho hệ thống xử lý dữ liệu:

### Bước 1: Thiết lập cấu hình đường dẫn
* Tạo các biến chứa đường dẫn của đối tượng nghiên cứu và thư mục dữ liệu cần thiết.
* Việc này giúp quản lý vị trí file một cách tập trung và dễ thay đổi khi di chuyển dự án.

### Bước 2: Chuyển đổi nhãn dữ liệu (Label Encoding)
* Đổi tên các hoạt động từ dạng chuỗi (String) sang dạng số (Integer).
* **Lý do:** Máy tính và các thuật toán học máy (Machine Learning) làm việc hiệu quả, tính toán nhanh hơn với con số thay vì chữ viết.

### Bước 3: Thu thập và chuẩn hóa danh sách file
* Sử dụng công cụ để thu thập danh sách tất cả các file dữ liệu `.csv`.
* Thực hiện sắp xếp chúng theo thứ tự bảng chữ cái để đảm bảo tính nhất quán trong quá trình xử lý lặp lại.

In [ ]:
# 2. Read and Combine all files
all_df = pd.DataFrame()
data_set = 1

for f in files:
    folder_name = os.path.basename(os.path.dirname(f))
    activity = folder_name.split("_")[0]
    trial = folder_name.split("_")[1]
    user_id = int(os.path.basename(f).replace("sub_", "").replace(".csv", ""))

    df = pd.read_csv(f)

    df["user_id"] = user_id
    df["label"] = LABEL_MAP[activity]
    df["set"] = data_set

    all_df = pd.concat([all_df, df], ignore_index=True)
    data_set += 1

## Quy trình Đọc và Gộp Dữ liệu (Data Collection & Combination)

Dưới đây là các bước chi tiết để hợp nhất nhiều file dữ liệu thô thành một bộ dữ liệu duy nhất:

---

### Bước 1: Khởi tạo và Trích xuất thông tin từ tên file
* **Khởi tạo:** Tạo một `DataFrame` rỗng (thường đặt tên là `all_df`) đóng vai trò là "thùng chứa" tổng.
* **Trích xuất:** Với mỗi file dữ liệu, đoạn code sẽ tự động phân tích đường dẫn và tên file để lấy ra các thông tin quan trọng (như loại hoạt động, số thứ tự lần thử nghiệm).

### Bước 2: Làm giàu dữ liệu (Data Enrichment)
Sau khi đọc nội dung từ mỗi file `.csv`, đoạn code sẽ chèn thêm **3 cột mới** vào bảng dữ liệu hiện tại để phục vụ việc định danh:
1.  **ID người tham gia (`user_id`):** Xác định dữ liệu này thuộc về ai.
2.  **Hoạt động (`label`):** Gán nhãn loại vận động dựa trên thư mục chứa file.
3.  **Số thứ tự file (`set`):** Lưu vết file đang được đọc để dễ dàng quản lý các phiên đo (sessions).

### Bước 3: Hợp nhất và Đánh chỉ mục (Merge & Reindex)
* **Gộp dữ liệu:** Sử dụng lệnh gộp (thường là `pd.concat`) để "dán" dữ liệu từ file vừa đọc vào cuối bảng tổng `all_df`.
* **Đánh số thứ tự hàng:** Thực hiện đánh lại chỉ số hàng (Reset Index) để đảm bảo các con số thứ tự không bị trùng lặp hoặc lộn xộn sau khi gộp nhiều file khác nhau.

---

In [ ]:
# 3. Integration: Merge with Subject Information

df_subjects = pd.read_csv(subjects_info_path)
df_subjects = df_subjects.rename(columns={'code': 'user_id'})

all_df = pd.merge(all_df, df_subjects, on='user_id', how='left')

## Quy trình Tích hợp Dữ liệu (Subject Data Integration)

Dưới đây là các bước để kết nối bảng dữ liệu cảm biến với bảng thông tin chi tiết của người tham gia:

---

### Bước 1: Chuẩn hóa cột định danh (Standardizing Join Key)
* **Xử lý:** Kiểm tra và đổi tên cột định danh trong bảng thông tin người dùng (ví dụ: đổi từ `code` thành `user_id`).
* **Mục đích:** Đảm bảo tên cột khớp hoàn toàn với cột `user_id` đã tạo ở các bước trước, giúp máy tính nhận diện được "chiếc cầu nối" giữa hai bảng dữ liệu.

### Bước 2: Gộp bảng thông minh (Left Join)
* **Cách vận hành:** Thực hiện phép gộp dữ liệu (`merge`) giữa bảng tổng và bảng thông tin người dùng dựa trên mã số ID.
* **Quy tắc ưu tiên (Left Join):** * Ưu tiên giữ lại **toàn bộ dữ liệu cảm biến** trong bảng bên trái (`all_df`).
    * Nếu `user_id` trùng nhau, các thông tin tương ứng (như tuổi, cân nặng, chiều cao) sẽ được ghép thêm vào hàng đó.
    * Đảm bảo không làm mất bất kỳ dữ liệu vận động nào đã thu thập được.

---

In [ ]:
# 4. Cleaning and Feature Construction

if "Unnamed: 0" in all_df.columns:
    del all_df["Unnamed: 0"]

all_df["acc_x"] = all_df["userAcceleration.x"] + all_df["gravity.x"]
all_df["acc_y"] = all_df["userAcceleration.y"] + all_df["gravity.y"]
all_df["acc_z"] = all_df["userAcceleration.z"] + all_df["gravity.z"]

all_df.rename(columns={
    'rotationRate.x': 'gyr_x',
    'rotationRate.y': 'gyr_y',
    'rotationRate.z': 'gyr_z'
}, inplace=True)

all_df.dropna(inplace=True)

all_df = all_df.drop(
    columns = ["userAcceleration.x", "userAcceleration.y", "userAcceleration.z", "gravity.x", "gravity.y", "gravity.z"]
)
all_df = all_df[
    [
        "acc_x", "acc_y", "acc_z",
        "gyr_x", "gyr_y", "gyr_z",
        "attitude.roll", "attitude.pitch", "attitude.yaw",
        "user_id", "label", "set",
        "weight", "height", "age", "gender"
    ]
]

## Giai đoạn: Làm sạch và Xây dựng đặc trưng (Cleaning & Feature Construction)

Quy trình này giúp chuẩn hóa dữ liệu thô thành các đặc trưng vật lý có ý nghĩa cho việc phân tích:

---

### Bước 1: Loại bỏ cột dư thừa
* Thực hiện xóa các cột chỉ mục không cần thiết (thường là cột `Unnamed: 0`) phát sinh trong quá trình lưu và đọc file CSV.

### Bước 2: Tạo đặc trưng gia tốc thực (Total Acceleration)
* **Thao tác:** Cộng giá trị `userAcceleration` với `gravity` để thu được các đặc trưng `acc_x`, `acc_y`, `acc_z`.
* **Lý do:** Dữ liệu thô từ điện thoại thường tách biệt giữa gia tốc do người dùng tạo ra và trọng lực của Trái Đất. Việc cộng lại giúp phản ánh chuyển động thực tế của người dùng trong không gian.

### Bước 3: Chuẩn hóa tên gọi cảm biến
* Đổi tên các cột thuộc con quay hồi chuyển (Gyroscope) thành các tên ngắn gọn, dễ hiểu và chuyên nghiệp hơn như: `gyr_x`, `gyr_y`, `gyr_z`.

### Bước 4: Tối ưu hóa bộ dữ liệu
* **Làm sạch:** Xóa bỏ tất cả các hàng có chứa giá trị `NaN` (dữ liệu trống) để tránh lỗi khi huấn luyện mô hình.
* **Tinh gọn:** Xóa các cột thành phần ban đầu (`userAcceleration`, `gravity`) sau khi đã tính toán xong gia tốc thực.
* **Lợi ích:** Giúp giảm dung lượng bộ nhớ và làm bộ dữ liệu gọn gàng hơn.

### Bước 5: Sắp xếp cấu trúc dữ liệu (Reordering)
Sắp xếp lại các cột trong bảng theo một trật tự logic để dễ dàng quan sát và truy xuất:
1. **Gia tốc** (`acc`)
2. **Con quay hồi chuyển** (`gyr`)
3. **Tư thế** (`Attitude`)
4. **Thông tin định danh** (`user_id`, `label`, `set`)
5. **Thông tin cá nhân** (Tuổi, giới tính, cân nặng, chiều cao)

---

In [ ]:
# 5. Working with Datetimes (50Hz = 20ms)

all_df["time_ms"] = all_df.groupby("set").cumcount() * 20
all_df.index = pd.to_datetime(all_df["time_ms"], unit="ms")

## Giai đoạn: Xử lý Thời gian (Working with Datetimes)

Quy trình này giúp thiết lập trục thời gian chính xác cho dữ liệu cảm biến, cho phép phân tích dữ liệu theo chuỗi thời gian:

---

### Bước 1: Xác định tần số lấy mẫu và Nhóm dữ liệu
* **Tần số:** Dữ liệu được thu thập ở mức **50Hz**, tương ứng với việc cứ mỗi **20ms** (mili giây) sẽ có một mẫu dữ liệu mới được ghi lại.
* **Nhóm dữ liệu:** Thực hiện nhóm dữ liệu theo từng `set` (phiên đo). Việc này đảm bảo thời gian sẽ bắt đầu lại từ 0 cho mỗi phiên đo mới của mỗi người dùng, tránh việc cộng dồn thời gian sai lệch giữa các tệp khác nhau.

### Bước 2: Tính toán mốc thời gian (Timestamp Calculation)
* **Thao tác:** Sử dụng hàm `cumcount()` để đánh số thứ tự các hàng trong mỗi nhóm, sau đó nhân với khoảng cách thời gian là **20ms**.
* **Kết quả:** Chuyển đổi số thứ tự hàng thành giá trị thời gian thực tế tương ứng tính bằng đơn vị mili giây (`time_ms`).

### Bước 3: Thiết lập chỉ mục thời gian (DatetimeIndex)
* **Chuyển đổi:** Chuyển đổi cột thời gian mili giây vừa tính toán sang định dạng `DatetimeIndex` (định dạng thời gian chuẩn của thư viện Pandas).
* **Gán Index:** Gán dãy thời gian này làm **Index** (chỉ mục) chính cho toàn bộ bảng dữ liệu.
* **Lợi ích:** Việc có `DatetimeIndex` cho phép sử dụng các công cụ phân tích mạnh mẽ như hàm `resample` (để thay đổi tần suất dữ liệu) hoặc dễ dàng vẽ đồ thị diễn biến hoạt động theo thời gian.

---